# Bounded compact BERT fine-tuning

## Objective and methodology

Reference the actual completed training run rather than rerun expensive optimization implicitly. The notebook validates the saved model configuration and observed loss history. Reproduce training with `python scripts/cache_transformer.py` followed by `python scripts/run_phase3.py --mode full`; the YAML config controls CPU/CUDA/MPS, batch size, accumulation and budgets.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd() if (Path.cwd()/"configs/phase3_config.yaml").exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT)); sys.path.insert(0,str(ROOT/"scripts"))
result = json.loads((ROOT/"reports/phase3_results.json").read_text())
assert result["mode"] == "full", "A smoke run is not a completed experiment"
print("Dataset sizes:", {k:v["reviews"] for k,v in result["dataset"]["splits"].items()})
print("All evaluation labels are rating-derived weak labels, not verified human sentiment.")

Dataset sizes: {'train': 18367, 'validation': 2489, 'test': 2451}
All evaluation labels are rating-derived weak labels, not verified human sentiment.


## Completed run and checkpoint validation

In [2]:
if "compact_bert" in result["models"]:
    from transformers import AutoConfig
    trained = result["models"]["compact_bert"]
    config = AutoConfig.from_pretrained(ROOT/trained["artifact_path"],local_files_only=True)
    assert config.num_labels == 3
    print("Actual device:",trained["device"],"completed epochs:",trained["epochs_completed"],"best epoch:",trained["best_epoch"])
    display(pd.DataFrame([{k:v for k,v in epoch.items() if k != "validation"} for epoch in trained["history"]]))
else:
    print("Transformer not trained:",result.get("transformer_failure"))

[project]/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Actual device: mps completed epochs: 4 best epoch: 3


,epoch,training_weighted_cross_entropy,validation_weighted_cross_entropy,elapsed_seconds
0,1,0.994506,0.772432,26.940997
1,2,0.710093,0.681828,53.051068
2,3,0.654131,0.669964,88.099908
3,4,0.626853,0.666412,116.714466


## Actual figures

External image links keep notebook outputs small.

In [3]:
figures = [name for name in result.get("figures",[]) if ("loss" in name if "06_" in '06_transformer_training' else "model_comparison" in name)]
for name in figures:
    display(Markdown("![Actual experimental figure](../reports/figures/"+name+")"))

![Actual experimental figure](../reports/figures/phase3_full_transformer_loss.png)

## Interpretation and limitations

This is an English compact-BERT resource pilot, not DistilBERT, XLM-R or proof of multilingual capability. The pilot uses a bounded training subset; checkpoint selection relies on validation macro F1, and losses are actual completed-epoch records.